# Part 2: Time Series Modeling

In this notebook, you will implement functions to extract features from time series data and build ARIMA models.

In [2]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from pathlib import Path
import os

# Set style for plots
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Feature Extraction

Implement the `extract_time_series_features` function to calculate rolling window features.

In [3]:
def extract_time_series_features(data, window_size=60):
    """Extract rolling window features from time series data.
    
    Parameters
    ----------
    data : pd.DataFrame
        Preprocessed physiological data
    window_size : int
        Size of the rolling window in seconds
        
    Returns
    -------
    pd.DataFrame
        DataFrame containing extracted features for each signal
    """
    # Your code here
    # Create a copy to avoid modifying the original data
    features = data.copy()

    # Define the rolling window in terms of samples
    window_samples = window_size

    # 1. Calculate rolling window statistics
    features['heart_rate_mean'] = features['heart_rate'].rolling(window=window_samples).mean()
    features['heart_rate_std'] = features['heart_rate'].rolling(window=window_samples).std()
    features['heart_rate_min'] = features['heart_rate'].rolling(window=window_samples).min()
    features['heart_rate_max'] = features['heart_rate'].rolling(window=window_samples).max()

    features['eda_mean'] = features['eda'].rolling(window=window_samples).mean()
    features['eda_std'] = features['eda'].rolling(window=window_samples).std()
    features['eda_min'] = features['eda'].rolling(window=window_samples).min()
    features['eda_max'] = features['eda'].rolling(window=window_samples).max()

    features['temperature_mean'] = features['temperature'].rolling(window=window_samples).mean()
    features['temperature_std'] = features['temperature'].rolling(window=window_samples).std()
    features['temperature_min'] = features['temperature'].rolling(window=window_samples).min()
    features['temperature_max'] = features['temperature'].rolling(window=window_samples).max()

    # 2. Autocorrelation (using lag 1)
    def autocorr(x):
        """Compute the autocorrelation for a series."""
        return np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else np.nan
    
    features['heart_rate_autocorr'] = features['heart_rate'].rolling(window=window_samples).apply(autocorr, raw=False)
    features['eda_autocorr'] = features['eda'].rolling(window=window_samples).apply(autocorr, raw=False)
    features['temperature_autocorr'] = features['temperature'].rolling(window=window_samples).apply(autocorr, raw=False)

    # Drop rows with NaN values that result from rolling window calculations
    features = features.dropna()

    return features

## 2. ARIMA Modeling

Implement the `build_arima_model` function to fit ARIMA models and generate diagnostic plots.

In [4]:
def build_arima_model(series, order=(1,1,1), output_dir='plots'):
    """Fit an ARIMA model to the time series and generate diagnostic plots.
    
    Parameters
    ----------
    series : pd.Series
        Time series data to model
    order : tuple
        (p,d,q) order of the ARIMA model
    output_dir : str
        Directory to save diagnostic plots
        
    Returns
    -------
    statsmodels.tsa.arima.model.ARIMAResults
        Fitted ARIMA model
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Fit ARIMA model
    model = ARIMA(series, order=order)
    results = model.fit()

    # 2. Generate diagnostic plots
    
    # Plot 1: Model fit (observed vs fitted)
    plt.figure(figsize=(10, 6))
    plt.plot(series, label='Observed', color='blue')
    plt.plot(results.fittedvalues, label='Fitted', color='red')
    plt.title('ARIMA Model Fit')
    plt.legend()
    plt.xlabel('Time')
    plt.ylabel('Value')
    model_fit_path = os.path.join(output_dir, 'model_fit.png')
    plt.savefig(model_fit_path)
    plt.close()
    
    # Plot 2: Residuals plot
    plt.figure(figsize=(10, 6))
    residuals = results.resid
    plt.plot(residuals, label='Residuals', color='green')
    plt.title('Residuals of ARIMA Model')
    plt.xlabel('Time')
    plt.ylabel('Residuals')
    residuals_path = os.path.join(output_dir, 'residuals.png')
    plt.savefig(residuals_path)
    plt.close()

    # Plot 3: Forecast plot (forecast next 10 points)
    forecast_steps = 10
    forecast = results.forecast(steps=forecast_steps)
    forecast_index = range(len(series), len(series) + forecast_steps)
    
    plt.figure(figsize=(10, 6))
    plt.plot(series, label='Observed', color='blue')
    plt.plot(forecast_index, forecast, label='Forecast', color='orange')
    plt.title('ARIMA Forecast')
    plt.xlabel('Time')
    plt.ylabel('Value')
    forecast_path = os.path.join(output_dir, 'forecast.png')
    plt.savefig(forecast_path)
    plt.close()

    # 3. Return the ARIMA results object
    return results